# Demo Step Function

A compact TensorFlow 2 / GPflow 2 version of the step-function regression demo.
It builds a two-layer DGP, runs a short optimizer step, and evaluates predictive moments.


In [ ]:
import numpy as np
import gpflow

gpflow.config.set_default_float(np.float64)
from gpflow.likelihoods import Gaussian
from gpflow.optimizers import Scipy
from gpflow.utilities import set_trainable

from doubly_stochastic_dgp.dgp import DGP

X = np.linspace(-1.0, 1.0, 24)[:, None]
Y = np.where(X < 0.0, -1.0, 1.0) + 0.05 * np.sin(20.0 * X)
Z = X[::4].copy()
kernels = [gpflow.kernels.SquaredExponential(), gpflow.kernels.SquaredExponential()]

model = DGP(X, Y, Z, kernels, Gaussian(), num_samples=3)
model.likelihood.likelihood.variance.assign(0.05)
set_trainable(model.likelihood.likelihood.variance, False)

loss_before = model.training_loss().numpy()
Scipy().minimize(model.training_loss, model.trainable_variables, options={"maxiter": 5})
mean, var = model.predict_y(X, 3)

float(loss_before), mean.shape, var.shape
